# Single GPU implementation

We start by converting the serial CPU version to a GPU-accelerated one using **managed memory**.

### 1. Memory Management

To use manage memory, adaptation of allocation and de-allocation functions is necessary.
We retain the concept of a single pointer, but now usable by host and device.
Data allocated like this will be **migrate** automatically on access between host and device - it as such removes the need for manual copies, e.g. via `cudaMemcpy`, and simplifies incremental ports.

* Replace `new[]` and `malloc` with `cudaMallocManaged`
* Replace `delete[]` and `free` with `cudaFree`

### 2. Convert Loop Nests

After revising memory handling, the next step is porting work to the GPU and parallelizing it there.
Our main workload is represented by the `stencil2D` (host) function which we convert to a **kernel**.

* Add the `__global__` keyword to denote a kernel
* Map threads to `i0` and `i1` tuples; use `blockIdx.x * blockDim.x + threadIdx.x` and `blockIdx.y * blockDim.y + threadIdx.y`
* Guard boundaries and against out-of-bounds accesses
* Compute one stencil update per thread, keeping the data layout identical (i.e. use the same **index linearization**)
* **Optional**: introduce grid-stride loops

### 3. Timing

For correct and accurate timings, we need to add synchronization points with the GPU as kernels are executed **asynchronously**.

* Use `cudaDeviceSynchronize()` to synchronize with the GPU **before starting** timing **before stopping** timing
* Remember to deactivate file output to get meaningful performance numbers

### 4. Prefetching for Performance

One optimization when using managed memory is to strategically and proactively move data ahead of time with `cudaMemPrefetchAsync`.

* Prefetch `u` and `uNew` to the GPU before the first iteration
* Prefetch `u` back to the CPU before post-processing (i.e. before computing the final temperature sum)
* Prefetch `u` to the CPU before printing to file and back to the GPU afterwards

## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/03-gpu-baseline/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Take the unmodified [02-cpu-baseline/stencil-2d.cpp](../src/02-cpu-baseline/stencil-2d.cpp) as base and port it to GPU.

In [ ]:
%%bash
if [ -e ../src/03-gpu-baseline/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/02-cpu-baseline/stencil-2d.cpp ../src/03-gpu-baseline/stencil-2d.cu
fi

### Level Medium

[stencil-2d-medium.cu](../src/03-gpu-baseline/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/03-gpu-baseline/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [1]:
%%bash
if [ -e ../src/03-gpu-baseline/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/03-gpu-baseline/stencil-2d-medium.cu ../src/03-gpu-baseline/stencil-2d.cu
fi

### Level Easier

[stencil-2d-easier.cu](../src/03-gpu-baseline/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/03-gpu-baseline/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [1]:
%%bash
if [ -e ../src/03-gpu-baseline/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/03-gpu-baseline/stencil-2d-easier.cu ../src/03-gpu-baseline/stencil-2d.cu
fi

### Possible Solution

[stencil-2d-solution.cu](../src/03-gpu-baseline/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/03-gpu-baseline/stencil-2d.cu) with the cell below.

In [ ]:
%%bash
if [ -e ../src/03-gpu-baseline/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/03-gpu-baseline/stencil-2d-solution.cu ../src/03-gpu-baseline/stencil-2d.cu
fi

### Compilation, Execution and Profiling

The new code version is available in [03-gpu-baseline/stencil-2d.cu](../src/03-gpu-baseline/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [3]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/03-gpu-baseline ../src/03-gpu-baseline/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [4]:
!../build/03-gpu-baseline 256 64 2 2000 100

  Completed iteration 0
  Completed iteration 100
  Completed iteration 200
  Completed iteration 300
  Completed iteration 400
  Completed iteration 500
  Completed iteration 600
  Completed iteration 700
  Completed iteration 800
  Completed iteration 900
  Completed iteration 1000
  Completed iteration 1100
  Completed iteration 1200
  Completed iteration 1300
  Completed iteration 1400
  Completed iteration 1500
  Completed iteration 1600
  Completed iteration 1700
  Completed iteration 1800
  Completed iteration 1900
  Completed iteration 2000
  #cells / #it:  16384 / 2000
  elapsed time:  181.834 ms
  per iteration: 0.0909171 ms
  MLUP/s:        180.208
  bandwidth:     2.88333 GB/s
  compute:       1.26146 GFLOP/s
  Total temperature is 0.406217


The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [5]:
!../build/03-gpu-baseline $((32 * 1024)) 256 2 16 0

  #cells / #it:  8388608 / 16
  elapsed time:  3.52343 ms
  per iteration: 0.220214 ms
  MLUP/s:        38093
  bandwidth:     609.488 GB/s
  compute:       266.651 GFLOP/s
  Total temperature is 1


Instead of running on the local **single A40** GPU, we can also submit a batch job running on a **single A100** GPU.

In [6]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:1 \
    --time 00:05:00 --wait \
    --output=../output/03-gpu-baseline.out --error=../output/03-gpu-baseline.err \
    --wrap="../build/03-gpu-baseline $((32 * 1024)) 256 2 8 0"

cat ../output/03-gpu-baseline.out

Submitted batch job 3428963
### Starting TaskPrologue of job 3428963 on a0701 at Tue Mar 10 11:14:00 CET 2026
Running on cores 0-15 with governor ondemand
Tue Mar 10 11:14:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:0E:00.0 Off |                    0 |
| N/A   36C    P0             53W /  400W |       0MiB /  40960MiB |      0% 

The next cell performs profiling with Nsight Systems by submitting a batch job.

The profile is then available at [profiles/03-gpu-baseline.nsys-rep](../profiles/03-gpu-baseline.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [7]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:1 \
    --time 00:05:00 --wait \
    --output=../output/03-gpu-baseline-nsys.out --error=../output/03-gpu-baseline-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/03-gpu-baseline \
        ../build/03-gpu-baseline $((32 * 1024)) 256 2 8 0"

cat ../output/03-gpu-baseline-nsys.out

Submitted batch job 3428964
### Starting TaskPrologue of job 3428964 on a0701 at Tue Mar 10 11:14:08 CET 2026
Running on cores 0-15 with governor ondemand
Tue Mar 10 11:14:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:0E:00.0 Off |                    0 |
| N/A   36C    P0             53W /  400W |       0MiB /  40960MiB |      0% 

## Next Step

The next step is adapting this implementation towards multi-GPU and multi-node scaling.
We start by splitting the work into multiple patches in notebook [04](./04-work-partitioning.ipynb).